# Phase 1 implementation blueprint: your actual Databricks environment
I'll tailor the implementation to your existing hrs_dbx_repo project rather than starting a new warehouse from scratch.
Your established environment and conventions are:
•	Workspace: Databricks Free Edition, serverless compute.
•	Runtime: client.5.12 for your existing notebook templates.
•	Repository: C:\AI Development\hrs_dbx_repo, using the feature branch and Databricks Asset Bundles.
•	Existing warehouse: dev_catalog, including brz_raw_hrs and slv_cdm_hrs.
•	Source data: RAND HRS longitudinal data converted from SPSS to partitioned Parquet and Delta.
•	Existing Silver model: cohort, wave, respondent, demographics, health, and future subject-area tables.
•	Deployment: GitHub Actions and your existing hrs_etl_job workflow.
We will extend these conventions and preserve the distinction between DDL, DML, validation, and orchestration.
Databricks Free Edition supports AI application and agent development, but it has limited serverless resources, a maximum of five concurrent job tasks, and other feature restrictions. We should therefore build a lightweight agent and verify each required feature in your workspace before relying on it. 
 

# Implementation roadmap
Phase 1 — Knowledge foundation
Build the Codebook metadata catalog and a searchable reference for HRS variables.
First milestone

Phase 2 — Warehouse discovery
Connect the agent to Unity Catalog and document existing tables, columns, keys, and relationships.

Phase 3 — Artifact generation
Generate version-controlled Silver and Gold DDL/DML specifications and SQL scripts.

Phase 4 — Validation and controlled execution
Test SQL, compare statistics with the Codebook, review proposed changes, and execute approved operations.

Phase 5 — Orchestration and continuous improvement
Generate workflow dependencies, log results, save approved decisions, and maintain regression tests.

# Phase 1 — Build the HRS Codebook knowledge foundation
The Codebook is the most important dependency. Without reliable variable definitions and source references, the agent could generate syntactically valid SQL that produces incorrect results.

## Step 1.1 — Organize the source documents
Create the following directories in your repository:
metadata/
├── codebook/
│   ├── source/
│   ├── extracted/
│   └── processing/
├── data_model/
└── approved_decisions/

Place the RAND HRS Codebook PDF in source/. Keep the original PDF unchanged; record its edition, filename, and processing status.

Use the PDF as the authoritative source for variable definitions and documented descriptive statistics. The extracted content will be a derived artifact that can be regenerated if extraction logic changes.

## Step 1.2 — Extract the Codebook

Create this notebook:
notebooks/agent/01_extract_hrs_codebook.ipynb

Its responsibilities are:
1.	Read the Codebook PDF.
2.	Extract text and page numbers.
3.	Identify variable-definition tables and descriptive-statistics tables.
4.	Preserve section headings and table context.
5.	Flag poorly extracted pages or records for review.
6.	Save the extracted output to a governed Delta schema.

Because the Codebook is large, process it page-by-page or in manageable batches rather than assuming the entire document will fit in one model prompt.
Do not let the LLM silently fill in missing or ambiguous definitions. Record uncertain extractions and require verification against the original page.

## Step 1.3 — Create a dedicated metadata schema

Use your existing catalog:
dev_catalog.hrs_agent_metadata
This is a proposed schema for the AI Agent, separate from slv_cdm_hrs and gld_star_hrs. Create it only after confirming that the name is available and that you have the necessary permissions.
Start with these metadata tables:
Table	Essential columns
hrs_variable_catalog	variable_name, variable_label, definition, survey_section, source_page, codebook_version
hrs_variable_wave_coverage	variable_name, wave_number, survey_year, coverage_status, source_page
hrs_variable_cohort_coverage	variable_name, hacohort_number, coverage_status, source_page
hrs_variable_statistics	variable_name, wave_number, hacohort_number, statistic_name, statistic_value, source_page
hrs_variable_value_labels	variable_name, value_code, value_label, source_page
hrs_document_chunks	document_id, page_number, section_heading, chunk_text, codebook_version
These are proposed starting schemas. Refine the keys and column types after inspecting the actual Codebook layout. Use DOUBLE for numeric statistical values when appropriate, and preserve categorical codes without assuming they are continuous numeric measures.
The variable catalog should distinguish the source variable name from the warehouse column name. For example, a source variable such as R1BMI should not be confused with a derived Gold measure such as average_bmi.

## Step 1.4 — Test the metadata with real questions
Before adding an LLM, verify that the extracted tables can answer these questions:
•	What does RACE mean?
•	Which variable represents gender or sex, according to the Codebook?
•	Which variable contains BMI for a given wave?
•	Which survey section contains a particular variable?
•	Which cohorts and waves document that variable?
•	What descriptive statistics does the Codebook report?

For each answer, return the supporting Codebook page or table reference. If a question is ambiguous, ask for the wave, cohort, or variable context required to answer it correctly.
Phase 1 acceptance criteria: The metadata is queryable in Delta, each tested answer has a traceable source reference, and uncertain extractions are flagged rather than guessed.

# Phase 2 — Connect the agent to Unity Catalog

## Step 2.1 — Discover the existing warehouse
Create notebooks/agent/02_inspect_hrs_catalog.ipynb.
The notebook should inspect:
•	Catalogs and schemas accessible to the current user.
•	Existing Silver and Gold tables.
•	Column names and data types.
•	Primary-key conventions and declared constraints.
•	Table and column comments, where available.
•	Row counts, null counts, and duplicate-key indicators.
•	Relationships inferred from approved specifications.

Start with read-only SQL using SHOW SCHEMAS, SHOW TABLES, DESCRIBE TABLE, and INFORMATION_SCHEMA views where supported.

For example:
SHOW SCHEMAS IN dev_catalog;

SHOW TABLES IN dev_catalog.slv_cdm_hrs;

DESCRIBE TABLE
    dev_catalog.slv_cdm_hrs.hrs_cohort;
These statements are for discovery; adjust them if a table name or schema differs in your current workspace.

## Step 2.2 — Create a machine-readable data model
Store the approved model specification in:
metadata/data_model/hrs_data_model.md
Include the following relationships from your existing design:
Additional fact tables should follow the same documented respondent/wave relationships when appropriate.
The agent must also know the intended grain of each fact table. For example, if fact_demographics stores one record per respondent per wave, its natural uniqueness rule should be documented and tested. Do not assume every fact table has the same grain.

## Step 2.3 — Create the agent's access model
Use Unity Catalog permissions to separate read-only discovery from write operations.
Access role	Permissions
Discovery	Read metadata and approved warehouse data.
Artifact author	Generate files and proposed SQL; no direct production changes.
Executor	Run explicitly approved DDL/DML within authorized schemas.
Validator	Read source, Silver, Gold, and validation results.
These can be logical roles in your design; you do not need to create four separate identities in Free Edition. Use the actual identities and grants supported by your workspace.
For a single-developer environment, begin with read-only discovery and manual SQL execution. Introduce automated write access only after validation is working.
Databricks supports Unity Catalog functions as governed agent tools, especially for structured data retrieval. Other logic can remain in Python when that is simpler. 
 

##  Phase 2 acceptance criteria: 
The agent can accurately list existing tables, explain the approved relationships, and distinguish actual catalog metadata from assumptions in the design.

# Phase 3 — Build the Python agent and artifact generator

## Step 3.1 — Implement a small set of Python modules
Extend your repository with:
agent/
├── agent.py
├── instructions/
│   └── system_prompt.md
├── tools/
│   ├── codebook_tools.py
│   ├── catalog_tools.py
│   ├── model_tools.py
│   ├── ddl_generator.py
│   ├── dml_generator.py
│   ├── validation_tools.py
│   ├── execution_tools.py
│   └── orchestration_tools.py
└── artifact_manager.py

Each module should have one responsibility. The agent should call tools with structured inputs and receive structured results.
For example, the DDL generator should receive:
•	Target layer and fully qualified table name.
•	Business purpose and table grain.
•	Columns and approved data types.
•	Primary-key and foreign-key conventions.
•	Source variable mappings.
•	Audit-column requirements.
•	Relevant existing specification.
•	Whether the request is a new table or a modification.

It should return proposed SQL, a summary of changes, dependencies, assumptions, and validation requirements.

## Step 3.2 — Use a structured artifact contract
Every generation request should produce a consistent artifact package.
Artifact package
DDL
sql/ddl/silver/ or sql/ddl/gold/
DML
sql/dml/silver/ or sql/dml/gold/
Validation SQL
sql/validation/
Orchestration specification
resources/ and the relevant notebook files
Change manifest
A machine-readable record of requirements, affected tables, dependencies, assumptions, approval status, and tests.
Use your existing template specifications as the standards for generated files. The agent should not generate a new naming convention every time.

## Step 3.3 — Enforce DDL and DML separation
The agent must respect the following rules:
•	DDL creates or modifies database objects.
•	DML loads and transforms data.
•	Validation SQL checks the correctness of those operations.
•	Orchestration controls execution order and failure handling.
Your existing execute_sql_file utility can be reused for SQL execution, with one important safeguard: retain the single-statement execution pattern that avoids passing multiple statements to one spark.sql() call.
For multi-statement scripts, use a reviewed SQL-aware statement parser rather than blindly splitting on semicolons, which can appear inside strings or SQL bodies.

## Step 3.4 — Add the system prompt
Create agent/instructions/system_prompt.md with instructions along these lines:
RAND HRS Data Warehouse Engineering Agent
RAND HRS Data Warehouse Engineering Agent
Role
You are a senior Databricks data engineer specializing in the RAND HRS Longitudinal dataset and the Databricks Medallion architecture.
Authoritative sources
1.	The RAND HRS Codebook is the authority for source-variable definitions, value labels, documented coverage, and published descriptive statistics.
2.	The current Unity Catalog is the authority for existing database objects and column definitions.
3.	Approved data model specifications define intended table grain, keys, relationships, and naming conventions.
4.	Approved repository artifacts define established DDL, DML, validation, and orchestration patterns.
Operating rules
•	Distinguish source variables from warehouse surrogate keys and derived measures.
•	Cite Codebook page references for variable definitions and coverage claims.
•	Do not invent variables, cohort applicability, wave coverage, keys, or table relationships.
•	Ask for clarification when a material requirement is ambiguous.
•	Inspect existing artifacts before proposing changes.
•	Generate DDL, DML, validation SQL, and orchestration as separate artifacts.
•	Explain the proposed changes and dependencies before execution.
•	Never execute unapproved destructive operations.
•	Validate generated SQL and data results before declaring success.
•	Record assumptions, validation outcomes, and approved decisions.
•	Treat failed validation as a failure requiring investigation, not as successful completion.

Business-question workflow
For each business question:
1.	Identify the required dimensions, measures, filters, and output grain.
2.	Map each attribute to verified HRS source variables and Codebook references.
3.	Identify the relevant Silver tables and join keys.
4.	Propose the Gold data model and table definitions.
5.	Generate DDL, DML, validation SQL, and orchestration dependencies.
6.	Present the change manifest for review.
7.	Execute only approved changes.
8.	validate the results and record the outcome.
9.	Save approved decisions for reuse in later requests.
Step 3.5 — Choose the agent runtime
Start with a Python agent running in a Databricks notebook. Use an available model endpoint and a supported tool-calling framework in your workspace.
A Databricks-native integration with Unity Catalog functions is a good option for governed data access. Confirm package compatibility, model availability, and serverless support before selecting the final runtime. 
 
## Phase 3 acceptance criteria: 
Given an approved table specification, the agent generates repeatable DDL/DML artifacts, identifies dependencies, and does not execute SQL merely because it generated it.

# Phase 4 — Implement data validation and controlled execution
This phase is where the agent moves from a code-generation assistant to a warehouse engineering assistant.

## Step 4.1 — Build a validation framework
Create notebooks/agent/03_validate_hrs_data.ipynb and sql/validation/.
The validation tool should support these categories:
Validation category	Example check
Codebook consistency	Compare documented variable statistics with Silver values.
Schema consistency	Verify expected columns and data types.
Key integrity	Check duplicate respondent/wave records at the intended grain.
Referential integrity	Find facts without a valid respondent or wave mapping.
Data quality	Check missing, invalid, or out-of-range values.
Transformation integrity	Confirm source-to-Silver row counts and documented filters.
Gold reconciliation	Recalculate selected aggregates independently and compare results.
Regression testing	Rerun previously approved tests after changes.
Do not compare statistics across mismatched populations. The validator must align wave, cohort, inclusion criteria, and special missing-value handling before reporting a discrepancy.

## Step 4.2 — Define the validation result schema
Create a table such as dev_catalog.hrs_agent_metadata.hrs_validation_results.
Recommended columns:
CREATE TABLE IF NOT EXISTS
    dev_catalog.hrs_agent_metadata.hrs_validation_results (
        validation_id STRING,
        run_id STRING,
        validation_timestamp TIMESTAMP,
        validation_type STRING,
        source_variable STRING,
        target_table STRING,
        target_column STRING,
        expected_value DOUBLE,
        actual_value DOUBLE,
        tolerance DOUBLE,
        status STRING,
        explanation STRING
    )
USING DELTA;
This is a starter DDL, not a complete audit model. Extend it for categorical statistics, row counts, minimum/maximum comparisons, source page references, and multiple metrics per validation.
For production-quality checks, preserve the exact source context and comparison method. A single numeric expected/actual pair is not sufficient for every test.

## Step 4.3 — Implement controlled execution
Use the following execution process for every proposed change.
The execution tool should:
•	Accept only approved artifact identifiers or approved SQL.
•	Verify the intended catalog and schema.
•	Reject unapproved destructive operations.
•	Record the artifact version and execution result.
•	Stop dependent tasks when validation fails.
•	Never report success merely because a SQL statement was submitted.
For your Free Edition environment, begin with manually approved execution from a notebook. Automate only the steps you have tested reliably.

## Phase 4 acceptance criteria: 
The agent can explain a validation failure, retain evidence of the failure, and prevent a failed change from being reported as successful.

# Phase 5 — Generate orchestration and enable continuous improvement

## Step 5.1 — Generate a dependency-aware workflow
Extend your existing hrs_etl_job instead of creating a second competing warehouse pipeline.
The conceptual workflow should be:
The Codebook extraction and agent evaluation can run as separate maintenance workflows. They should not be repeated unnecessarily on every data load.
In the existing Asset Bundle configuration, define task dependencies explicitly and keep each task's responsibility narrow. Your current five-concurrent-task account limit means the workflow should be designed around sequential dependencies rather than unnecessary parallelism. 
 
Databricks on AWS
Step 5.2 — Persist approved knowledge
Create a decision log containing:
•	Requirement identifier and description.
•	Approved business definition.
•	Source-variable mappings and Codebook references.
•	Affected Silver and Gold objects.
•	Approved DDL/DML artifact versions.
•	Validation results.
•	Approval and execution status.
•	Superseded decisions and the reason for the change.
When the user corrects a definition, the agent should propose an update to the relevant knowledge record. It should not silently overwrite an established definition.

## Step 5.3 — Evaluate the agent after each change
Create notebooks/agent/04_evaluate_hrs_agent.ipynb.
Maintain a test set containing questions such as:
1.	What does RACE mean?
2.	Which HRS variable identifies BMI for the requested wave?
3.	Which cohorts and waves contain a particular variable?
4.	Which Silver table contains a requested measure?
5.	Generate Gold DDL for a specified business question.
6.	Generate DML using the approved source-to-target mappings.
7.	Identify a mismatch between Silver descriptive statistics and the Codebook.
8.	Propose a data model change without modifying the existing warehouse.
9.	Explain why a validation failed.
10.	Generate an orchestration plan that stops when validation fails.
For each test, record the expected evidence, actual answer, correctness, and any regression. The agent is improving only when approved knowledge and tests become more reliable—not simply because its prompts or conversation history grow.
6. Your first development sprint
I recommend completing these tasks in order before attempting the full BMI use case.

Implementation checklist
0 of 7 complete
Confirm workspace capabilities
Verify model endpoint access, serverless notebook execution, Unity Catalog permissions, and any available document retrieval features.
Create the agent folders
Add the agent, metadata, SQL, notebook, and test directories to hrs_dbx_repo.
Make the Codebook available
Place the PDF in the project source location and verify that the notebook can read it.
Extract and validate Codebook metadata
Implement notebook 01 and verify variable definitions, page references, and statistics on a small sample.
Inspect the current catalog
Implement notebook 02 and capture the real table definitions and relationships.
Build the first read-only agent
Support Codebook questions and catalog lookups before enabling artifact generation or execution.
Test with the BMI use case
Require verified source mappings, a proposed Gold grain, DDL/DML, and validation tests.
Copy checklist 

7. What I would build first
The first concrete deliverable should be a working notebook named:
notebooks/agent/01_extract_hrs_codebook.ipynb
It should extract a small, verifiable sample of the Codebook, create the first metadata Delta tables, and demonstrate several successful variable lookups. That gives us a foundation we can test before adding SQL generation or write permissions.
One important design decision: keep the agent's knowledge, the generated artifacts, and the live warehouse state separate. The Codebook explains the source data; approved specifications define the intended design; Unity Catalog reveals what currently exists; and validation establishes whether the implementation is correct.
That separation is what will let your agent evolve as requirements change without losing control over the integrity of your RAND HRS data warehouse.

